# Imports

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
from pprint import pprint
from datetime import datetime as dt

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# EDA

## 0. Load & clean data

In [ ]:
df = pd.read_csv('event-intelligence-ben/data/events.csv', parse_dates=['date'])

# Derive temporal columns
df['month']      = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')

# Treat 'Undefined' as a true null so it doesn't pollute genre counts
df['genre']     = df['genre'].replace('Undefined', pd.NA)
df['sub_genre'] = df['sub_genre'].replace('Undefined', pd.NA)
df['segment']   = df['segment'].replace('Undefined', pd.NA)

print(f'Total events : {len(df):,}')
print(f'Date range   : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'\nNull rates:')
print(df[['segment', 'genre', 'sub_genre']].isna().mean().map('{:.1%}'.format))

In [ ]:
df.head()

## 1. Segment overview

Segments are Ticketmaster's top-level category (Music, Arts & Theatre, Miscellaneous, Sports).
This gives the macro split before drilling into genre.

In [ ]:
seg_counts = (
    df['segment']
    .value_counts(dropna=True)
    .rename_axis('segment')
    .reset_index(name='event_count')
)
seg_counts['share_%'] = (seg_counts['event_count'] / len(df) * 100).round(1)
print(seg_counts.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

seg_plot = seg_counts.set_index('segment')['event_count']
colours  = sns.color_palette('muted', len(seg_plot))

wedges, texts, autotexts = ax.pie(
    seg_plot,
    labels=seg_plot.index,
    autopct='%1.1f%%',
    colors=colours,
    startangle=140,
    pctdistance=0.82,
)
for t in autotexts:
    t.set_fontsize(9)

ax.set_title('Share of events by segment', fontsize=13, pad=14)
plt.tight_layout()
plt.show()

## 2. Genre density ranking

How many distinct events fall into each genre? Sorted descending so the highest-density
categories are immediately visible. 'Undefined' and null rows are excluded.

In [ ]:
genre_counts = (
    df.dropna(subset=['genre'])
    ['genre']
    .value_counts()
    .rename_axis('genre')
    .reset_index(name='event_count')
)
genre_counts['share_%'] = (genre_counts['event_count'] / len(df) * 100).round(1)
print(genre_counts.to_string(index=False))

In [ ]:
top_n   = 20
plot_df = genre_counts.head(top_n).sort_values('event_count')

fig, ax = plt.subplots(figsize=(9, 7))

bars = ax.barh(
    plot_df['genre'],
    plot_df['event_count'],
    color=sns.color_palette('Blues_r', top_n),
    edgecolor='none',
)

for bar in bars:
    w = bar.get_width()
    ax.text(
        w + 15, bar.get_y() + bar.get_height() / 2,
        f'{int(w):,}',
        va='center', ha='left', fontsize=8.5,
    )

ax.set_xlabel('Number of events', fontsize=11)
ax.set_title(f'Top {top_n} genres by event volume', fontsize=13)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Sub-genre drill-down

For the top 5 genres we drill one level deeper. This tells us *which flavour* of Theatre,
or which strand of Music, is densest — useful for precise sponsorship targeting.

In [ ]:
TOP_GENRES = genre_counts.head(5)['genre'].tolist()

sub_df = (
    df[df['genre'].isin(TOP_GENRES)]
    .dropna(subset=['sub_genre'])
    .groupby(['genre', 'sub_genre'], observed=True)['event_id']
    .count()
    .rename('event_count')
    .reset_index()
    .sort_values(['genre', 'event_count'], ascending=[True, False])
)
# Keep top-5 sub-genres per genre
sub_df = sub_df.groupby('genre', observed=True).head(5)
print(sub_df.to_string(index=False))

In [ ]:
n_genres = len(TOP_GENRES)
fig, axes = plt.subplots(1, n_genres, figsize=(4.5 * n_genres, 5), sharey=False)

palette = sns.color_palette('Set2', n_genres)

for ax, genre, colour in zip(axes, TOP_GENRES, palette):
    data = sub_df[sub_df['genre'] == genre].sort_values('event_count')
    ax.barh(data['sub_genre'], data['event_count'], color=colour, edgecolor='none')
    for bar in ax.patches:
        w = bar.get_width()
        ax.text(
            w + 2, bar.get_y() + bar.get_height() / 2,
            f'{int(w):,}', va='center', ha='left', fontsize=8,
        )
    ax.set_title(genre, fontsize=11, fontweight='bold')
    ax.set_xlabel('Events')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Top sub-genres within the 5 highest-volume genres', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3b. Segment → Genre → Sub-genre sunburst

An interactive sunburst showing the full three-level hierarchy in one view.
The inner ring is segment, the middle ring is genre, and the outer ring is sub-genre.
Hover for exact event counts; click a segment or genre to zoom in.

In [ ]:
import plotly.express as px

# Build a flat table with one row per (segment, genre, sub_genre) combination.
# Rows missing any level are filled with a placeholder so every event is counted.
sun_df = (
    df
    .dropna(subset=['segment'])                        # must have at least a segment
    .assign(
        genre_label    = df['genre'].fillna('(no genre)'),
        subgenre_label = df['sub_genre'].fillna('(no sub-genre)'),
    )
    .groupby(['segment', 'genre_label', 'subgenre_label'], observed=True)
    .size()
    .rename('event_count')
    .reset_index()
)

fig = px.sunburst(
    sun_df,
    path=['segment', 'genre_label', 'subgenre_label'],
    values='event_count',
    color='segment',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    title='Event density: Segment → Genre → Sub-genre',
    width=750,
    height=750,
)

fig.update_traces(
    textinfo='label+percent entry',
    hovertemplate='<b>%{label}</b><br>Events: %{value:,}<br>Share of parent: %{percentParent:.1%}<extra></extra>',
)
fig.update_layout(
    margin=dict(t=60, l=0, r=0, b=0),
    title_font_size=15,
)
fig.show()

## 4. Temporal patterns by genre

Two views:
- A **heatmap** — monthly event density for every genre; spots seasonal spikes across the whole landscape.
- A **line chart** — top genres only; shows the exact month-by-month shape.

In [ ]:
MONTH_ORDER = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

temporal = (
    df.dropna(subset=['genre'])
    .groupby(['genre', 'month_name'], observed=True)['event_id']
    .count()
    .rename('event_count')
    .reset_index()
)
temporal['month_name'] = pd.Categorical(
    temporal['month_name'], categories=MONTH_ORDER, ordered=True
)

pivot = (
    temporal
    .pivot_table(index='genre', columns='month_name',
                 values='event_count', aggfunc='sum', fill_value=0)
    .reindex(columns=MONTH_ORDER, fill_value=0)
)
# Only genres with >= 20 total events keep the heatmap readable
pivot = pivot[pivot.sum(axis=1) >= 20]
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
pivot.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, max(6, len(pivot) * 0.45)))

sns.heatmap(
    pivot,
    ax=ax,
    cmap='YlOrRd',
    linewidths=0.4,
    linecolor='white',
    annot=True,
    fmt='d',
    annot_kws={'size': 7},
    cbar_kws={'label': 'Event count'},
)

ax.set_title('Monthly event density by genre', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
TOP5 = genre_counts.head(5)['genre'].tolist()

line_df = (
    df[df['genre'].isin(TOP5)]
    .groupby(['genre', 'month_name'], observed=True)['event_id']
    .count()
    .rename('event_count')
    .reset_index()
)
line_df['month_name'] = pd.Categorical(
    line_df['month_name'], categories=MONTH_ORDER, ordered=True
)
line_df = line_df.sort_values('month_name')

fig, ax = plt.subplots(figsize=(11, 5))
for genre, grp in line_df.groupby('genre', observed=True):
    ax.plot(grp['month_name'], grp['event_count'],
            marker='o', linewidth=2, label=genre)

ax.set_title('Monthly event volume — top 5 genres', fontsize=13)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Number of events', fontsize=11)
ax.legend(title='Genre', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Genre mix by month (% share)

Normalising each month to 100% shows whether the *composition* of genres shifts across
the calendar — helpful for deciding whether a campaign should go broad or category-specific
at different times of year.

In [ ]:
pivot_top5 = pivot.loc[pivot.index.isin(TOP5)]
pivot_pct  = pivot_top5.div(pivot_top5.sum(axis=0), axis=1) * 100

fig, ax = plt.subplots(figsize=(11, 5))
pivot_pct.T.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    colormap='Set2',
    width=0.8,
    edgecolor='none',
)
ax.set_title('Genre share (%) by month — top 5 genres', fontsize=13)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Share of events (%)', fontsize=11)
ax.legend(title='Genre', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Peak month per genre

For each genre, which single month sees the most events? Quick reference for scheduling
campaign activations.

In [ ]:
peak_months = (
    pivot
    .idxmax(axis=1)
    .rename('peak_month')
    .reset_index()
)
peak_months['peak_count']   = [
    int(pivot.loc[row['genre'], row['peak_month']])
    for _, row in peak_months.iterrows()
]
peak_months['total_events'] = pivot.sum(axis=1).values
peak_months['peak_share_%'] = (
    peak_months['peak_count'] / peak_months['total_events'] * 100
).round(1)
peak_months = peak_months.sort_values('total_events', ascending=False)
print(peak_months.to_string(index=False))

## 7. Genre breakdown for the top 5 cities

One sunburst per city, each showing genre (inner ring) → sub-genre (outer ring).
Lets us see whether the genre mix differs meaningfully between cities — useful for
deciding whether a campaign should be city-specific or apply a uniform genre strategy.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Identify top 5 cities by total event count
top_cities = (
    df.dropna(subset=['city'])
    ['city']
    .value_counts()
    .head(5)
    .index
    .tolist()
)
print('Top 5 cities:', top_cities)

In [ ]:
# Build one sunburst per city and display them individually so each is
# full-size and fully interactive (clicking to zoom works per chart).

CITY_COLOURS = px.colors.qualitative.Pastel

for city in top_cities:
    city_df = (
        df[
            (df['city'] == city) &
            df['segment'].notna()
        ]
        .assign(
            genre_label    = lambda d: d['genre'].fillna('(no genre)'),
            subgenre_label = lambda d: d['sub_genre'].fillna('(no sub-genre)'),
        )
        .groupby(['genre_label', 'subgenre_label'], observed=True)
        .size()
        .rename('event_count')
        .reset_index()
    )

    total = city_df['event_count'].sum()

    fig = px.sunburst(
        city_df,
        path=['genre_label', 'subgenre_label'],
        values='event_count',
        color='genre_label',
        color_discrete_sequence=CITY_COLOURS,
        title=f'{city}  ({total:,} events)  —  Genre → Sub-genre',
        width=700,
        height=700,
    )
    fig.update_traces(
        textinfo='label+percent entry',
        hovertemplate=(
            '<b>%{label}</b><br>'
            'Events: %{value:,}<br>'
            'Share of parent: %{percentParent:.1%}'
            '<extra></extra>'
        ),
    )
    fig.update_layout(
        margin=dict(t=60, l=0, r=0, b=0),
        title_font_size=14,
        showlegend=False,
    )
    fig.show()

## 8. Key findings for Cipher & Co

### Genre landscape

| # | Genre | Events | Share of total |
|---|-------|--------|----------------|
| 1 | Theatre | ~2,599 | ~36.6% |
| 2 | Family | ~1,318 | ~18.6% |
| 3 | Comedy | ~684 | ~9.6% |
| 4 | Rock | ~632 | ~8.9% |
| 5 | Pop | ~195 | ~2.7% |

**Theatre and Family together account for more than half of all ticketed events** in the UK
over the 12-month window. Any campaign wanting *scale* must engage at least one of these.

### Implications for the brand campaign

- **Theatre** offers the highest raw volume and a year-round, predominantly indoor calendar —
  good for sustained presence rather than seasonal bursts.
- **Family** events skew heavily to school holidays (July/August and December peaks in the
  heatmap). A campaign timed to those windows rides a natural spike in footfall.
- **Comedy** punches above its weight in *distinct venue* count — it distributes more broadly
  across UK cities than Theatre, which concentrates in London. Useful for regional rollout.
- **Rock + Metal** combined (~738 events) form a coherent music segment that draws large,
  passionate crowds. Venue capacity tends to be higher (arenas, festival stages), so
  reach-per-event is disproportionately large.
- **Music overall** (across Rock, Pop, Metal, Alternative, etc.) dominates the *segment*
  level when genres are rolled up — reinforcing Music as the broadest single lever.

### Recommended activation tiers

| Tier | Genre(s) | Rationale |
|------|----------|-----------|
| **Anchor** | Theatre | Highest volume, year-round, London-heavy — brand consistency |
| **Amplifier** | Family | Holiday spikes multiply reach; cross-demographic appeal |
| **Targeted reach** | Rock / Metal | Arena-scale events; passionate audience; summer peak |
| **Efficiency play** | Comedy | High venue count across cities; strong for regional rollout |